# FlagOS Challenge: `median.dim` Operator — v2 (Optimized)
**Operator #11 — Medium Difficulty — Reduction Category**

Implements `torch.median(input, dim, keepdim=False) -> (values, indices)` using Triton.

### Algorithm (v2 — Correct Approach)
`torch.median` internally uses **sorting** then indexing at position `N // 2`.

Our strategy mirrors this exactly:
1. **Sort** along `dim` using `torch.sort` (batched Radix Sort via CUDA — identical to what FlagGems `sort_stable` wraps).
2. **Gather** using a tiny O(M) Triton kernel that reads `sorted[..., N//2]` and its index.

This should produce speedup ≈ 1.0x (matching PyTorch) with potential wins from the fused gather.

> **Why v1 was 0.18x:** Bitonic Sort is O(N log²N) and runs in a single thread block. PyTorch uses O(N) `nth_element` or a parallel radix sort. Our new approach uses the same sort backend.

In [ ]:
!pip install -q triton torch

In [ ]:
import torch
import triton
import triton.language as tl
import time
from collections import namedtuple

print(f"PyTorch: {torch.__version__}")
print(f"Triton:  {triton.__version__}")
print(f"CUDA:    {torch.cuda.get_device_name(0)}")

PyTorch: 2.10.0+cu128
Triton:  3.6.0
CUDA:    Tesla T4


## 1. Triton Gather Kernel
After sorting, we just need to read position `N // 2` per row. This is O(M) and very fast.

In [ ]:
@triton.jit
def median_gather_kernel(
    sorted_val_ptr,
    sorted_idx_ptr,
    out_val_ptr,
    out_idx_ptr,
    N,
    BLOCK_M: tl.constexpr,
):
    """
    Each block of BLOCK_M programs reads the element at position N//2
    from the presorted (M, N) arrays and writes to the output (M,) arrays.

    O(M) total reads — extremely fast after sorting.
    """
    pid = tl.program_id(0)
    row_ids = pid * BLOCK_M + tl.arange(0, BLOCK_M)
    # median position: lower of the two middle elements for even N
    median_pos = N // 2

    # Each row reads exactly one element
    src_offsets = row_ids * N + median_pos

    v = tl.load(sorted_val_ptr + src_offsets)
    i = tl.load(sorted_idx_ptr + src_offsets)

    tl.store(out_val_ptr + row_ids, v)
    tl.store(out_idx_ptr + row_ids, i)


def triton_median_dim(inp: torch.Tensor, dim: int, keepdim: bool = False):
    """
    Triton implementation of torch.median(input, dim, keepdim).

    Returns namedtuple(values, indices) matching PyTorch's interface.

    Pipeline:
      1. torch.sort along dim (fast parallel Radix Sort via CUDA)
      2. Triton gather kernel picks position N//2 per row
    """
    assert inp.is_floating_point(), "median only supports floating-point tensors"
    if dim < 0:
        dim = dim + inp.ndim

    # Move reduction dim to the last position → (M, N) layout
    if dim != inp.ndim - 1:
        work = torch.movedim(inp, dim, -1).contiguous()
    else:
        work = inp.contiguous()

    N = work.shape[-1]
    M = work.numel() // N

    # Step 1: Sort along last dim (same as torch.median does internally)
    sorted_vals, sorted_idxs = torch.sort(work, dim=-1, stable=True)

    # Step 2: Triton gather — pick position N//2 per row
    out_shape = list(work.shape[:-1])  # (M,) or multi-dim
    out_val = torch.empty(out_shape, dtype=work.dtype, device=inp.device)
    out_idx = torch.empty(out_shape, dtype=torch.int64, device=inp.device)

    BLOCK_M = min(256, triton.next_power_of_2(M))
    grid = (triton.cdiv(M, BLOCK_M),)
    median_gather_kernel[grid](
        sorted_vals, sorted_idxs,
        out_val, out_idx,
        N,
        BLOCK_M=BLOCK_M,
    )

    if keepdim:
        out_val = out_val.unsqueeze(dim)
        out_idx = out_idx.unsqueeze(dim)

    Median = namedtuple('median', ['values', 'indices'])
    return Median(values=out_val, indices=out_idx)


print('Optimized median kernel ready.')

Optimized median kernel ready.


## 2. Correctness Tests

In [ ]:
def test_correctness():
    print('=' * 20, 'CORRECTNESS TESTS', '=' * 20)
    all_pass = True

    test_cases = [
        ('1D small (N=5)',     (5,),          0,  False),
        ('1D large (N=1001)', (1001,),        0,  False),
        ('2D cols (N=7)',     (32, 7),        1,  False),
        ('2D rows (N=32)',    (32, 7),        0,  False),
        ('3D last dim',       (4, 8, 64),     2,  False),
        ('3D mid dim',        (4, 8, 64),     1,  False),
        ('keepdim=True',      (16, 33),       1,  True),
        ('Large (1024,512)',  (1024, 512),    1,  False),
        ('Even N=100',        (64, 100),      1,  False),
        ('N=1 (degenerate)',  (32, 1),        1,  False),
    ]

    for desc, shape, dim, kd in test_cases:
        x = torch.randn(shape, dtype=torch.float32, device='cuda')

        ref = torch.median(x, dim=dim, keepdim=kd)
        our = triton_median_dim(x, dim=dim, keepdim=kd)

        val_match = torch.allclose(our.values, ref.values, atol=1e-5)
        idx_match = torch.all(our.indices == ref.indices).item()
        ok = val_match and idx_match
        if not ok: all_pass = False

        v = '✓' if val_match else '✗'
        i = '✓' if idx_match else '✗'
        status = 'PASS' if ok else 'FAIL'
        print(f'  [{status}] {desc:<28} val={v}  idx={i}')

    print()
    print('ALL TESTS PASSED ✓' if all_pass else '⚠ SOME TESTS FAILED')

test_correctness()

==================== CORRECTNESS TESTS ====================
  [PASS] 1D small (N=5)               val=✓  idx=✓
  [PASS] 1D large (N=1001)            val=✓  idx=✓
  [PASS] 2D cols (N=7)                val=✓  idx=✓
  [FAIL] 2D rows (N=32)               val=✗  idx=✗
  [FAIL] 3D last dim                  val=✗  idx=✗
  [FAIL] 3D mid dim                   val=✗  idx=✗
  [PASS] keepdim=True                 val=✓  idx=✓
  [FAIL] Large (1024,512)             val=✗  idx=✗
  [FAIL] Even N=100                   val=✗  idx=✗
  [PASS] N=1 (degenerate)             val=✓  idx=✓

⚠ SOME TESTS FAILED


## 3. Performance Benchmark
Expected: ≈ 1.0x (matching PyTorch) — same sort backend, cheaper gather.

In [ ]:
def benchmark():
    print('\n' + '=' * 20, 'PERFORMANCE v2', '=' * 20)
    print(f"{'Shape':<20} {'dim':<5} {'Triton(us)':>12} {'PyTorch(us)':>12} {'Speedup':>9}")
    print('-' * 65)

    def run_bench(fn, *args, n_warmup=50, n_iter=200, **kwargs):
        for _ in range(n_warmup): fn(*args, **kwargs)
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(n_iter): fn(*args, **kwargs)
        torch.cuda.synchronize()
        return (time.perf_counter() - t0) / n_iter * 1e6

    configs = [
        ((1024, 64),    1),
        ((1024, 512),   1),
        ((4096, 64),    1),
        ((4096, 512),   1),
        ((1024, 1024),  1),
        ((512, 2048),   1),
    ]

    for shape, dim in configs:
        x = torch.randn(shape, device='cuda')
        t_ours = run_bench(triton_median_dim, x, dim)
        t_ref  = run_bench(torch.median, x, dim=dim)
        spd = t_ref / t_ours
        ok = '✓' if spd >= 0.9 else '✗'
        print(f"  {str(shape):<20} {dim:<5} {t_ours:>12.1f} {t_ref:>12.1f} {spd:>8.2f}x {ok}")

    print('\n✓ = meets competition threshold (≥ 0.9x)')

benchmark()


==================== PERFORMANCE v2 ====================
Shape                dim     Triton(us)  PyTorch(us)   Speedup
-----------------------------------------------------------------
  (1024, 64)           1            404.2         37.6     0.09x ✗
  (1024, 512)          1            499.6        249.0     0.50x ✗
  (4096, 64)           1            401.6         42.6     0.11x ✗
  (4096, 512)          1            914.7        425.3     0.46x ✗
  (1024, 1024)         1            344.1        237.8     0.69x ✗
  (512, 2048)          1            379.0        165.3     0.44x ✗

✓ = meets competition threshold (≥ 0.9x)


## 4. Timing Breakdown
Show how much time is spent in Sort vs Gather.

In [ ]:
def time_breakdown():
    print('\n' + '=' * 20, 'TIMING BREAKDOWN', '=' * 20)
    shape = (1024, 512)
    x = torch.randn(shape, device='cuda')
    N = shape[1]
    M = shape[0]

    def run_n(fn, n=200):
        for _ in range(50): fn()
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(n): fn()
        torch.cuda.synchronize()
        return (time.perf_counter() - t0) / n * 1e6

    # Sort step
    t_sort = run_n(lambda: torch.sort(x, dim=-1, stable=True))

    # Gather step only
    sv, si = torch.sort(x, dim=-1, stable=True)
    ov = torch.empty(M, dtype=x.dtype, device=x.device)
    oi = torch.empty(M, dtype=torch.int64, device=x.device)
    BLOCK_M = 256
    grid = (triton.cdiv(M, BLOCK_M),)
    t_gather = run_n(lambda: median_gather_kernel[grid](sv, si, ov, oi, N, BLOCK_M=BLOCK_M))

    # Full pipeline
    t_total = run_n(lambda: triton_median_dim(x, dim=1))

    print(f"  Sort step:       {t_sort:>8.1f} µs")
    print(f"  Gather step:     {t_gather:>8.1f} µs")
    print(f"  Total pipeline:  {t_total:>8.1f} µs")
    print(f"  Gather overhead: {t_gather/t_total*100:.1f}% of total")

time_breakdown()


==================== TIMING BREAKDOWN ====================
  Sort step:          230.1 µs
  Gather step:         23.2 µs
  Total pipeline:     366.4 µs
  Gather overhead: 6.3% of total
